# 08 · Zaman Serisi Tahmini
Geçmiş değerlere bakarak geleceği tahmin etmek (satış, talep, borsa, enerji tüketimi).

> ⚠️ Zaman serisinde veriyi **asla rastgele karıştırmayın**; kronolojik olarak ayırın (geçmiş → eğitim, gelecek → test).

**ARIMA(p, d, q)**: AR (geçmiş değerler) + I (fark alma ile durağanlaştırma) + MA (geçmiş hatalar).
**SARIMA** buna mevsimsel bileşen `(P, D, Q, s)` ekler.

Veri: Trend + 12 aylık mevsimsellik içeren sentetik aylık satış serisi (10 yıl).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error

In [ ]:
rng = np.random.default_rng(42)
t = np.arange(120)
seri = pd.Series(100 + 0.5*t + 10*np.sin(2*np.pi*t/12) + rng.normal(0, 3, 120),
                 index=pd.date_range("2016-01-01", periods=120, freq="MS"), name="satis")
print("Eksik değer:", seri.isna().sum())
seri.plot(figsize=(12, 4), color="tab:blue", title="Aylık Satış"); plt.show()

**Yorum:** Seride yukarı yönlü bir **trend** ve her yıl tekrarlanan **mevsimsel** dalgalanma açıkça görülüyor.

## 1. Ayrıştırma ve durağanlık

In [ ]:
seasonal_decompose(seri, model="additive", period=12).plot().set_size_inches(12, 8); plt.show()

In [ ]:
def adf(s, ad):
    p = adfuller(s.dropna())[1]
    print(f"{ad:25s} ADF p-değeri = {p:.4f} → {'DURAĞAN' if p < 0.05 else 'durağan DEĞİL'}")
adf(seri, "Orijinal seri")
adf(seri.diff(), "1. fark")
adf(seri.diff().diff(12), "1. fark + mevsimsel fark")

**Yorum:** Orijinal seri trend nedeniyle durağan değil. Fark alındığında durağan hale gelir → ARIMA'da `d=1` kullanacağız.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(seri.diff().dropna(), lags=36, ax=ax[0]); plot_pacf(seri.diff().dropna(), lags=36, ax=ax[1])
plt.tight_layout(); plt.show()

**Yorum:** ACF'de 12, 24, 36. gecikmelerdeki belirgin çubuklar **12 aylık mevsimselliği** doğrular → SARIMA'da `s=12`.

## 2. Kronolojik ayırma ve modeller

In [ ]:
train, test = seri[:-12], seri[-12:]          # son 12 ay test

naif = pd.Series(train.iloc[-12:].values, index=test.index)   # baseline: geçen yılın aynı ayı
arima = ARIMA(train, order=(1, 1, 1)).fit()
sarima = ARIMA(train, order=(1, 1, 1), seasonal_order=(1, 0, 1, 12)).fit()

tahminler = {"Naif (geçen yıl)": naif, "ARIMA(1,1,1)": arima.forecast(12), "SARIMA": sarima.forecast(12)}
pd.Series({ad: mean_absolute_error(test, p) for ad, p in tahminler.items()}, name="Test MAE").round(2)

In [ ]:
plt.figure(figsize=(13, 4.5))
train[-36:].plot(label="Train", color="tab:gray")
test.plot(label="Gerçek", color="black", lw=2)
for (ad, p), c in zip(tahminler.items(), ["tab:orange", "tab:red", "tab:green"]):
    p.plot(label=ad, ls="--", color=c)
plt.legend(); plt.title("Son 12 ay tahminleri"); plt.show()

**Yorum:** Mevsimselliği bilmeyen ARIMA düz bir çizgi tahmin ederek dalgaları kaçırır. SARIMA hem trendi hem mevsimselliği yakalar ve en düşük MAE'yi verir. Naif baseline'ı geçemeyen bir model kullanılmamalıdır.

In [ ]:
# Son model tüm veriyle yeniden eğitilir ve gelecek 12 ay tahmin edilir
final = ARIMA(seri, order=(1, 1, 1), seasonal_order=(1, 0, 1, 12)).fit()
gelecek = final.get_forecast(12)
ci = gelecek.conf_int()
plt.figure(figsize=(13, 4.5)); seri[-36:].plot(label="Geçmiş", color="tab:blue")
gelecek.predicted_mean.plot(label="Gelecek tahmini", color="tab:green")
plt.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], color="tab:green", alpha=0.2, label="%95 güven aralığı")
plt.legend(); plt.title("Gelecek 12 Ay"); plt.show()

## Özet
### Data Analysis Key Findings
- Seri trend ve 12 aylık mevsimsellik içeriyor; 1. fark ile durağan hale geliyor.
- SARIMA, mevsimselliği modellemeyen ARIMA'ya ve naif baseline'a göre daha düşük test hatası verdi.
### Insights or Next Steps
- Alternatifler: **Prophet**, gecikme (lag) özellikleriyle **XGBoost**, **LSTM**.
- Gerçek veride tatil, kampanya gibi dış değişkenleri SARIMAX'in `exog` parametresiyle ekleyin.